# This notebook analyzes SwimVision angle distributions and exports a learned pro distribution.

# SwimVision Analysis

This notebook loads processed keypoints, computes phase-wise angle statistics by swimmer level, plots overlapping histograms, and writes `results/pro_distribution.json`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.metrics.joint_angles import compute_all_angles

LABELS_PATH = PROJECT_ROOT / 'data' / 'labels.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(LABELS_PATH)
labels_df.head()

In [ ]:
PHASE_WINDOWS = {
    'block_phase': ('block_start', 'block_end'),
    'flight_phase': ('flight_start', 'flight_end'),
    'entry_phase': ('entry_start', 'entry_end'),
}
PHASE_METRICS = {
    'block_phase': ['front_knee_angle', 'rear_knee_angle', 'hip_angle', 'torso_lean'],
    'flight_phase': ['body_linearity', 'entry_angle', 'elbow_extension'],
    'entry_phase': ['streamline_angle', 'elbow_lock_angle'],
}

def infer_phase_boundaries_from_length(total_frames: int) -> dict[str, int]:
    split_one = max(int(total_frames * 0.33), 1)
    split_two = max(int(total_frames * 0.66), split_one + 1)
    return {
        'block_start': 0,
        'block_end': split_one - 1,
        'flight_start': split_one,
        'flight_end': split_two - 1,
        'entry_start': split_two,
        'entry_end': total_frames - 1,
    }

def load_clip_phase_stats(clip_id: str) -> dict[str, dict[str, float]]:
    keypoints_path = PROCESSED_DIR / f'{clip_id}_keypoints.npy'
    keypoints = np.load(keypoints_path)
    angles_df = compute_all_angles(keypoints)
    phase_boundaries = infer_phase_boundaries_from_length(len(angles_df))
    clip_stats: dict[str, dict[str, float]] = {}
    for phase_name, (start_key, end_key) in PHASE_WINDOWS.items():
        window = angles_df.loc[phase_boundaries[start_key]:phase_boundaries[end_key]]
        clip_stats[phase_name] = {}
        for metric in PHASE_METRICS[phase_name]:
            if metric == 'body_linearity':
                clip_stats[phase_name][metric] = float(window[metric].max())
            elif metric in {'entry_angle', 'elbow_extension'}:
                if metric == 'elbow_extension':
                    clip_stats[phase_name][metric] = float(
                        max(window.iloc[-1]['left_elbow_angle'], window.iloc[-1]['right_elbow_angle'])
                    )
                else:
                    clip_stats[phase_name][metric] = float(window.iloc[-1][metric])
            elif metric == 'elbow_lock_angle':
                clip_stats[phase_name][metric] = float(
                    window[['left_elbow_angle', 'right_elbow_angle']].mean(axis=1).mean()
                )
            else:
                clip_stats[phase_name][metric] = float(window[metric].mean())
    return clip_stats


In [ ]:
records = []
for row in labels_df.to_dict(orient='records'):
    clip_id = row['clip_id']
    keypoints_path = PROCESSED_DIR / f'{clip_id}_keypoints.npy'
    if not keypoints_path.exists():
        continue
    clip_stats = load_clip_phase_stats(clip_id)
    for phase_name, metrics in clip_stats.items():
        for metric_name, value in metrics.items():
            records.append({
                'clip_id': clip_id,
                'level': row['level'],
                'phase': phase_name,
                'metric': metric_name,
                'value': value,
            })

analysis_df = pd.DataFrame(records)
analysis_df.head()

In [ ]:
summary_df = (
    analysis_df.groupby(['level', 'phase', 'metric'])['value']
    .agg(['mean', 'std'])
    .reset_index()
    .sort_values(['phase', 'metric', 'level'])
)
summary_df

In [ ]:
colors = {'pro': 'blue', 'competitive': 'orange', 'novice': 'red'}
for phase_name, metrics in PHASE_METRICS.items():
    for metric_name in metrics:
        plt.figure(figsize=(8, 4))
        for level_name in ['pro', 'competitive', 'novice']:
            subset = analysis_df[(analysis_df['phase'] == phase_name) & (analysis_df['metric'] == metric_name) & (analysis_df['level'] == level_name)]
            if subset.empty:
                continue
            plt.hist(subset['value'], bins=15, alpha=0.45, label=level_name, color=colors[level_name])
        plt.title(f'{phase_name}: {metric_name}')
        plt.xlabel('Angle / metric value')
        plt.ylabel('Frequency')
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
pro_distribution: dict[str, dict[str, tuple[float, float]]] = {}
for phase_name, metrics in PHASE_METRICS.items():
    pro_distribution[phase_name] = {}
    for metric_name in metrics:
        subset = analysis_df[(analysis_df['phase'] == phase_name) & (analysis_df['metric'] == metric_name) & (analysis_df['level'] == 'pro')]
        if subset.empty:
            pro_distribution[phase_name][metric_name] = [0.0, 0.0]
        else:
            pro_distribution[phase_name][metric_name] = [
                float(subset['value'].mean()),
                float(subset['value'].std(ddof=0) if len(subset) > 1 else 0.0),
            ]

output_path = RESULTS_DIR / 'pro_distribution.json'
with open(output_path, 'w', encoding='utf-8') as handle:
    json.dump(pro_distribution, handle, indent=2)

output_path